# Simplified Copy-Trade Backtest

Simulates copying trades of a hand-picked set of wallets against the
processed Polygon trade shards.

## Strategy
- When a watched wallet prints a **BUY** on token `(condition_id, outcome)` at price `p`
  we place a buy order at a **slightly worse** price: `order_price = clip(p + WORSE_PRICE_OFFSET, 0.001, 0.999)`.
- The order is matched against the **first subsequent trade** whose effective price is
  `<= order_price` within `FILL_HORIZON_SECONDS`.
  - *same-token* liquidity: later BUY trades on the same `(condition_id, outcome)`.
  - *opposite-token* liquidity: later SELL trades on the **complementary** outcome
    (binary markets only), with effective price = `1 - sell_price`.
- For a wallet **SELL** the mirror applies: order at `p - WORSE_PRICE_OFFSET`, filled
  by the first later trade with effective price `>= order_price`.
  - same-token: later SELL prints on the same outcome.
  - opposite-token: later BUY prints on the complementary outcome, effective price `1 - buy_price`.

## Sharding
Trades are partitioned by the first hex character of `condition_id` (after `0x`).
All shards are processed in parallel; within each shard the backtest is exact
because a `condition_id` always falls in exactly one shard file.

## Outputs
One event ledger DataFrame with columns:
- `row_type` — `trigger` (watched-wallet trade that generated an order) or `fill` (our fill).
- `order_id` — unique integer identifying the copy order.
- `wallet` — originating wallet (`fill` rows carry the trigger wallet for reference).
- `condition_id`, `outcome`, `side`, `dt`, `tx_hash`, `price`.
- `order_price` — the price our order was placed at (`NaN` for fill rows).
- `fill_source` — `same_token` / `opposite_token` / `NaN` for trigger rows.
- `token_winner` — market resolution flag.
- `fill_pnl_usdc` — PnL of *our* position on fill rows (NaN for trigger rows),
  computed as stake × ( 1/exec_price − 1 ) for winning tokens, −stake for losers.
- `filled_wallet_total_position` — total filled position for this wallet across all conditions/outcomes.
- `filled_token_total_position` — total filled position for this `(condition_id, outcome)` across all wallets.
- `filled_token_by_wallet_position` — filled position for this `(wallet, condition_id, outcome)`.

## Visualisation
Cumulative PnL chart with two series:
- **Wallet cohort** — raw `trade_pnl` of the watched wallets.
- **Copy strategy** — `fill_pnl_usdc` of our simulated fills.


## Configuration

In [272]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

# ── Input wallets ─────────────────────────────────────────────────────────────
# Replace with any wallet addresses you want to copy.
# Example: load from a CSV, a workspace registry, or define inline.
WATCHED_WALLETS: list[str] = None

ONLY_BUY_TRIGGERS = False
MAX_EXPOSURE_PER_WALLET_1h = 200

# ── Test period ───────────────────────────────────────────────────────────────
# None = use entire dataset (start/end are inferred from the data).
# Set to datetime.date objects to restrict the window.
PERIOD_START: datetime.date | None = datetime.date(2026, 3, 16)
# PERIOD_START: datetime.date | None = datetime.date(2026, 1, 1) # None   # e.g. datetime.date(2026, 2, 16)
PERIOD_END:   datetime.date | None = datetime.date(2026, 5, 20)
# PERIOD_END:   datetime.date | None = datetime.date(2026, 2, 16) # None   # e.g. datetime.date(2026, 3, 11)

# ── Copy-trade execution parameters ──────────────────────────────────────────
WORSE_PRICE_OFFSET: float = 0
FILL_HORIZON_SECONDS: float = 300     # max seconds after trigger to search for a fill
STAKE_USDC: float = 10.0               # max USDC per order (order qty = min(trigger_qty, STAKE_USDC / order_price))
FEE_BPS: float = 10.0                   # taker fee in basis points
MAX_POSITION_PER_CONDITION: float | None = 1000  # max qty per (wallet, condition_id) across all outcomes; None = unlimited
MAX_POSITION_PER_WALLET:    float | None = 100  # max qty per wallet across all conditions; None = unlimited

# ── Data ─────────────────────────────────────────────────────────────────────
TRADES_DIR     = Path("../../data/polygon_trades_processed")
RAW_TRADES_DIR = Path("../../data/trades_polygon")

# ── Parallelism ───────────────────────────────────────────────────────────────
MAX_WORKERS: int = 10   # number of threads for parallel shard processing

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [273]:
banned_wallets = set()
if(WATCHED_WALLETS is not None):
    WATCHED_WALLETS = [w for w in WATCHED_WALLETS if w not in banned_wallets]

## Optionally load wallets from stage-1 workspace

If `WATCHED_WALLETS` is empty above, this cell loads the wallet set from the
stage-1 strategy registry. Skip or modify as needed.

In [274]:
import pandas as pd

if not WATCHED_WALLETS:
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
    wallets_path = WORKSPACE_DIR / "selected_wallets_v2.parquet"
    if wallets_path.exists():
        _wallets_df = pd.read_parquet(wallets_path, columns=["wallet"])
        WATCHED_WALLETS = _wallets_df["wallet"].tolist()
        print(f"Loaded {len(WATCHED_WALLETS)} wallets from {wallets_path.name}")
    else:
        print("No wallet source found — set WATCHED_WALLETS manually or run stage1 first.")
else:
    print(f"Using {len(WATCHED_WALLETS)} manually specified wallets.")

Loaded 131 wallets from selected_wallets_v2.parquet


## Discover shards and derive test period

In [275]:
import pandas as pd

SHARD_PATHS: list[Path] = sorted(TRADES_DIR.glob("*.parquet"))
print(f"Found {len(SHARD_PATHS)} shards under {TRADES_DIR}")

# Derive END_DATE_TRAIN from the is_train flag (last date where is_train=True).
# Test data is everything strictly after END_DATE_TRAIN.
_sample = pd.read_parquet(SHARD_PATHS[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN: datetime.date = _sample.loc[_sample["is_train"], "dt"].max().date()
_data_end: datetime.date      = _sample["dt"].max().date()
del _sample
print(f"END_DATE_TRAIN (last train date) : {END_DATE_TRAIN}")

# Backtest always runs on test data only (strictly after END_DATE_TRAIN).
# PERIOD_START / PERIOD_END can narrow the window further, but cannot go
# earlier than the day after END_DATE_TRAIN.
#_test_start = END_DATE_TRAIN + datetime.timedelta(days=1)
period_start: datetime.date = PERIOD_START # datetime.date(2026, 2, 16) #  max(PERIOD_START, _test_start) if PERIOD_START is not None else _test_start
period_end:   datetime.date = PERIOD_END #if PERIOD_END is not None else _data_end
print(f"Backtest period : {period_start}  →  {period_end}")

Found 16 shards under ../../data/polygon_trades_processed
END_DATE_TRAIN (last train date) : 2026-03-15
Backtest period : 2026-03-16  →  2026-05-20


## Backtest engine (per-shard)

Each shard is processed independently:
1. Load only rows within the backtest period (test data only).
2. Identify trigger events (watched-wallet trades).
3. Build a per-`condition_id` opposite-outcome map (binary markets only).
4. Process triggers in time order, maintaining a copy-portfolio **position** per `(wallet, condition_id, outcome)`.
5. For each trigger, place a copy order:
   - **BUY**: `qty_ordered = min(trig_qty, STAKE_USDC / order_price)` — worst-case loss = `qty × order_price ≤ STAKE_USDC`
   - **SELL**: `qty_ordered = min(trig_qty, position, STAKE_USDC / (1 − order_price))` — worst-case loss = `qty × (1 − order_price) ≤ STAKE_USDC`. Skipped if position = 0 (no short selling).
6. Scan tape prints in time order within `FILL_HORIZON_SECONDS`. Each matching print partially fills the order: `fill_qty = min(remaining_qty, tape_qty)`.
7. One **fill** ledger row is emitted per partial fill. BUY fills increment position; SELL fills decrement it.

`fill_pnl_usdc` per fill row (all executed at `exec_price = order_price`, limit fill):
- **BUY winner**: `fill_qty × (1 − exec_price) − fill_qty × exec_price × fee`
- **BUY loser**:  `−fill_qty × exec_price × (1 + fee)`
- **SELL winner** (sold below par): `fill_qty × (exec_price − 1) − fill_qty × exec_price × fee`
- **SELL loser** (sold above par):  `fill_qty × exec_price × (1 − fee)`


In [276]:
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _build_complement_map(df: pd.DataFrame) -> dict[tuple[str, str], str]:
    """Return {(condition_id, outcome): opposite_outcome} for binary markets."""
    pairs: dict[str, set] = {}
    for cid, grp in df.groupby("condition_id", sort=False):
        pairs[cid] = set(grp["outcome"].dropna().unique())
    result: dict[tuple[str, str], str] = {}
    for cid, outcomes in pairs.items():
        if len(outcomes) == 2:
            a, b = sorted(outcomes)
            result[(cid, a)] = b
            result[(cid, b)] = a
    return result


def _simulate_shard(
    shard_path: Path,
    raw_tape_path: Path,
    watched_wallets: set[str],
    period_start: datetime.date,
    period_end: datetime.date,
    worse_price_offset: float,
    fill_horizon_seconds: float,
    stake_usdc: float,
    fee_bps: float,
    max_position_per_condition: float | None = None,
    max_position_per_wallet: float | None = None,
) -> pd.DataFrame:
    """Process one shard file and return an event ledger DataFrame.

    Triggers are read from ``shard_path`` (processed per-wallet aggregated trades),
    which supplies ``token_winner`` and ``trade_pnl``.

    The fill tape is read from ``raw_tape_path`` (raw on-chain individual fills).
    Orders are simulated chronologically against the tape so that each tape print's
    quantity can only be consumed once across all active copy orders in the shard.
    """
    TRIG_COLS = [
        "wallet", "condition_id", "outcome", "dt", "side",
        "avg_price", "total_quantity", "trade_pnl", "token_winner",
        "tx_hash",
    ]
    tdf = pd.read_parquet(shard_path, columns=TRIG_COLS)
    if tdf.empty:
        return pd.DataFrame()

    tdf["dt"] = pd.to_datetime(tdf["dt"], utc=True)
    tdf = tdf.rename(columns={"avg_price": "price", "total_quantity": "quantity"})

    date_mask = (
        (tdf["dt"].dt.date >= period_start)
        & (tdf["dt"].dt.date <= period_end)
    )
    tdf = tdf[date_mask].copy()
    if tdf.empty:
        return pd.DataFrame()

    tdf["price"] = tdf["price"].astype(float)
    tdf["quantity"] = tdf["quantity"].astype(float)

    if ONLY_BUY_TRIGGERS:
        trigger_mask = (tdf["wallet"].isin(watched_wallets)) & (tdf["side"] == "BUY")
    else:
        trigger_mask = tdf["wallet"].isin(watched_wallets)
    triggers_df = tdf[trigger_mask].copy()
    if triggers_df.empty:
        return pd.DataFrame()

    tw_map: dict[tuple[str, str], bool] = {}
    for row in tdf[["condition_id", "outcome", "token_winner"]].itertuples(index=False):
        key = (row.condition_id, row.outcome)
        if key not in tw_map and row.token_winner is not None:
            tw_map[key] = bool(row.token_winner)

    complement = _build_complement_map(tdf)
    triggers_df = triggers_df.sort_values("dt", kind="mergesort").reset_index(drop=True)

    TAPE_COLS = ["condition_id", "outcome", "block_timestamp", "side", "price", "quantity", "tx_hash", "token_winner"]
    if raw_tape_path.exists():
        rdf = pd.read_parquet(raw_tape_path, columns=TAPE_COLS)
    else:
        rdf = pd.DataFrame(columns=TAPE_COLS)

    if rdf.empty:
        ledger_rows = []
        for trig in triggers_df.itertuples(index=False):
            cid = trig.condition_id
            outcome = trig.outcome
            side = trig.side
            trig_dt = trig.dt
            trig_px = float(trig.price)
            trig_qty = float(trig.quantity)
            trig_tw = bool(trig.token_winner)
            wallet = trig.wallet
            if side == "BUY":
                order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
                qty_ordered = min(trig_qty, stake_usdc / order_px)
            else:
                order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
                qty_ordered = trig_qty
            ledger_rows.append({
                "row_type": "trigger", "order_id": len(ledger_rows) + 1,
                "wallet": wallet, "condition_id": cid, "outcome": outcome,
                "side": side, "dt": trig_dt, "tx_hash": trig.tx_hash,
                "price": trig_px, "order_price": order_px,
                "qty_ordered": qty_ordered, "qty_filled": 0.0,
                "fill_qty": None, "fill_source": None,
                "token_winner": trig_tw, "exec_price": None,
                "fill_pnl_usdc": None,
                "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
                "filled_wallet_total_position": 0.0,
                "filled_token_total_position": 0.0,
                "filled_token_by_wallet_position": 0.0,
            })
        result = pd.DataFrame(ledger_rows)
        result["shard"] = shard_path.stem
        return result

    rdf["dt"] = pd.to_datetime(rdf["block_timestamp"], unit="s", utc=True)
    rdf = rdf.drop(columns=["block_timestamp"])

    tape_start = pd.Timestamp(period_start, tz="UTC")
    tape_end = pd.Timestamp(period_end, tz="UTC") + pd.Timedelta(days=1)
    rdf = rdf[(rdf["dt"] >= tape_start) & (rdf["dt"] < tape_end)].copy()
    rdf["price"] = rdf["price"].astype(float)
    rdf["quantity"] = rdf["quantity"].astype(float)
    rdf = rdf.sort_values("dt", kind="mergesort").reset_index(drop=True)

    fee = fee_bps / 10_000.0
    horizon = pd.Timedelta(seconds=fill_horizon_seconds)
    eps = 1e-12

    ledger_rows: list[dict] = []
    order_counter = 0
    # (wallet, condition_id, outcome) → filled qty for this outcome
    positions: dict[tuple[str, str, str], float] = defaultdict(float)
    # (wallet, condition_id) → total filled qty across all outcomes of that condition
    cond_positions: dict[tuple[str, str], float] = defaultdict(float)
    # (condition_id, outcome) → total filled qty for this outcome across all wallets
    token_total_positions: dict[tuple[str, str], float] = defaultdict(float)
    # wallet → total filled qty across all conditions
    wallet_positions: dict[str, float] = defaultdict(float)

    orders: dict[int, dict] = {}
    books: dict[tuple[str, str, str], list[dict]] = defaultdict(list)

    def _append_trigger_row(
        order_id: int,
        trig,
        order_px: float,
        qty_ordered: float,
        trig_tw: bool,
        current_token_pos: float = 0.0,
        current_token_total_pos: float = 0.0,
        current_wallet_pos: float = 0.0,
    ) -> None:
        ledger_rows.append({
            "row_type": "trigger",
            "order_id": order_id,
            "wallet": trig.wallet,
            "condition_id": trig.condition_id,
            "outcome": trig.outcome,
            "side": trig.side,
            "dt": trig.dt,
            "tx_hash": trig.tx_hash,
            "price": float(trig.price),
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "qty_filled": None,
            "fill_qty": None,
            "fill_source": None,
            "token_winner": trig_tw,
            "exec_price": None,
            "fill_pnl_usdc": None,
            "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
            "filled_wallet_total_position": current_wallet_pos,
            "filled_token_total_position": current_token_total_pos,
            "filled_token_by_wallet_position": current_token_pos,
        })

    # order can match with trade on the same side and token, or opposite side and opposite token    
    def _register_order(order_id: int, order: dict, trig_tw: bool) -> None:
        cid = order["condition_id"]
        outcome = order["outcome"]
        side = order["side"]
        books[(cid, outcome, side)].append({
            "order_id": order_id,
            "fill_source": "same_token",
            "fill_tw": trig_tw,
            "opposite": False,
        })
        opp_outcome = complement.get((cid, outcome))
        if opp_outcome is None:
            return
        opp_tw = tw_map.get((cid, opp_outcome), not trig_tw)
        opp_tape_side = "SELL" if side == "BUY" else "BUY"
        books[(cid, opp_outcome, opp_tape_side)].append({
            "order_id": order_id,
            "fill_source": "opposite_token",
            "fill_tw": trig_tw,
            "opposite": True,
        })

    def _process_tape_row(row) -> None:
        key = (row.condition_id, row.outcome, row.side)
        entries = books.get(key)
        if not entries:
            return

        tape_dt = row.dt
        tape_price = float(row.price)
        tape_remaining = float(row.quantity)
        if tape_remaining <= eps:
            return

        survivors: list[dict] = []
        for entry in entries:
            order = orders.get(entry["order_id"])
            if order is None:
                continue
            if order["remaining_qty"] <= eps:
                continue
            if order["deadline"] < tape_dt:
                continue

            eff_price = float(np.clip(1.0 - tape_price, 0.001, 0.999)) if entry["opposite"] else tape_price
            eligible = eff_price <= order["order_price"] if order["side"] == "BUY" else eff_price >= order["order_price"]

            if eligible and tape_remaining > eps:
                fill_qty = min(order["remaining_qty"], tape_remaining)

                # Cap fill to remaining position headroom (per-condition and per-wallet limits).
                # This prevents overfill when a single tape print is large enough to exceed the cap.
                if order["side"] == "BUY":
                    if max_position_per_condition is not None:
                        cond_headroom = max(max_position_per_condition - cond_positions[(order["wallet"], order["condition_id"])], 0.0)
                        fill_qty = min(fill_qty, cond_headroom)
                    if max_position_per_wallet is not None:
                        wallet_headroom = max(max_position_per_wallet - wallet_positions[order["wallet"]], 0.0)
                        fill_qty = min(fill_qty, wallet_headroom)
                    if fill_qty <= eps:
                        # Position cap already reached — retire this order without consuming tape
                        order["remaining_qty"] = 0.0
                        continue

                tape_remaining -= fill_qty
                order["remaining_qty"] -= fill_qty

                if order["side"] == "BUY":
                    gross = fill_qty * (1.0 - order["order_price"]) if entry["fill_tw"] else -fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    positions[order["pos_key"]] += fill_qty
                    cond_positions[(order["wallet"], order["condition_id"])] += fill_qty
                    token_total_positions[(order["condition_id"], order["outcome"])] += fill_qty
                    wallet_positions[order["wallet"]] += fill_qty
                    new_token_pos = positions[order["pos_key"]]
                    new_token_total_pos = token_total_positions[(order["condition_id"], order["outcome"])]
                    new_wallet_pos = wallet_positions[order["wallet"]]
                else:
                    gross = fill_qty * (order["order_price"] - 1.0) if entry["fill_tw"] else fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    prev = positions[order["pos_key"]]
                    positions[order["pos_key"]] = max(prev - fill_qty, 0.0)
                    actually_reduced = prev - positions[order["pos_key"]]
                    cond_positions[(order["wallet"], order["condition_id"])] = max(
                        cond_positions[(order["wallet"], order["condition_id"])] - actually_reduced, 0.0
                    )
                    wallet_positions[order["wallet"]] = max(
                        wallet_positions[order["wallet"]] - actually_reduced, 0.0
                    )
                    token_total_positions[(order["condition_id"], order["outcome"])] = max(
                        token_total_positions[(order["condition_id"], order["outcome"])] - actually_reduced, 0.0
                    )
                    new_token_pos = positions[order["pos_key"]]
                    new_token_total_pos = token_total_positions[(order["condition_id"], order["outcome"])]
                    new_wallet_pos = wallet_positions[order["wallet"]]

                ledger_rows.append({
                    "row_type": "fill",
                    "order_id": entry["order_id"],
                    "wallet": order["wallet"],
                    "condition_id": order["condition_id"],
                    "outcome": order["outcome"],
                    "side": order["side"],
                    "dt": tape_dt,
                    "tx_hash": row.tx_hash,
                    "price": eff_price,
                    "order_price": None,
                    "qty_ordered": order["qty_ordered"],
                    "qty_filled": None,
                    "fill_qty": fill_qty,
                    "fill_source": entry["fill_source"],
                    "token_winner": entry["fill_tw"],
                    "exec_price": order["order_price"],
                    "fill_pnl_usdc": fill_pnl,
                    "wallet_trade_pnl": None,
                    "filled_wallet_total_position": new_wallet_pos,
                    "filled_token_total_position": new_token_total_pos,
                    "filled_token_by_wallet_position": new_token_pos,
                })

            if order["remaining_qty"] > eps and order["deadline"] >= tape_dt:
                survivors.append(entry)

        if survivors:
            books[key] = survivors
        else:
            books.pop(key, None)

    tape_iter = iter(rdf[["condition_id", "outcome", "dt", "side", "price", "quantity", "tx_hash", "token_winner"]].itertuples(index=False))
    current_tape = next(tape_iter, None)

    for trig in triggers_df.itertuples(index=False):
        while current_tape is not None and current_tape.dt <= trig.dt:
            _process_tape_row(current_tape)
            current_tape = next(tape_iter, None)

        cid = trig.condition_id
        outcome = trig.outcome
        side = trig.side
        trig_px = float(trig.price)
        trig_qty = float(trig.quantity)
        trig_tw = bool(trig.token_winner)
        wallet = trig.wallet
        pos_key = (wallet, cid, outcome)

        if side == "BUY":
            order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
            current_pos = positions.get(pos_key, 0.0)
            qty_ordered = min(trig_qty, stake_usdc / order_px)
        else:
            order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
            current_pos = positions.get(pos_key, 0.0)
            if current_pos <= eps:
                continue
            sell_cap = stake_usdc / max(1.0 - order_px, 1e-9)
            qty_ordered = min(trig_qty, current_pos, sell_cap)
            if qty_ordered <= eps:
                continue

        order_counter += 1
        order_id = order_counter

        order = {
            "wallet": wallet,
            "condition_id": cid,
            "outcome": outcome,
            "side": side,
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "remaining_qty": qty_ordered,
            "deadline": trig.dt + horizon,
            "pos_key": pos_key,
        }
        orders[order_id] = order

        _append_trigger_row(
            order_id,
            trig,
            order_px,
            qty_ordered,
            trig_tw,
            current_token_pos=positions.get(pos_key, 0.0),
            current_token_total_pos=token_total_positions.get((cid, outcome), 0.0),
            current_wallet_pos=wallet_positions.get(wallet, 0.0),
        )
        _register_order(order_id, order, trig_tw)

    while current_tape is not None:
        _process_tape_row(current_tape)
        current_tape = next(tape_iter, None)

    if not ledger_rows:
        return pd.DataFrame()

    result = pd.DataFrame(ledger_rows)
    result["shard"] = shard_path.stem

    filled_by_order = (
        result[result["row_type"] == "fill"]
        .groupby("order_id")["fill_qty"]
        .sum()
        .rename("_total_filled")
    )
    result = result.merge(filled_by_order, on="order_id", how="left")
    trig_mask = result["row_type"] == "trigger"
    result.loc[trig_mask, "qty_filled"] = result.loc[trig_mask, "_total_filled"].fillna(0.0)
    result = result.drop(columns=["_total_filled"])
    result["qty_filled"] = result["qty_filled"].astype(float)

    return result



## Run backtest across all shards (parallel)

In [277]:
assert WATCHED_WALLETS, "WATCHED_WALLETS is empty — set wallets in the config cell or run the wallet-load cell."

watched_set = set(WATCHED_WALLETS)
shard_results: list[pd.DataFrame] = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _simulate_shard,
            path,
            RAW_TRADES_DIR / path.name,
            watched_set,
            period_start,
            period_end,
            WORSE_PRICE_OFFSET,
            FILL_HORIZON_SECONDS,
            STAKE_USDC,
            FEE_BPS,
            MAX_POSITION_PER_CONDITION,
            MAX_POSITION_PER_WALLET,
        ): path
        for path in SHARD_PATHS
    }
    for future in as_completed(futures):
        path = futures[future]
        try:
            df = future.result()
            if df is not None and not df.empty:
                shard_results.append(df)
        except Exception as exc:
            print(f"Shard {path.name} raised an exception: {exc}")
            raise

if shard_results:
    event_ledger: pd.DataFrame = (
        pd.concat(shard_results, ignore_index=True)
        .sort_values(["dt", "shard", "order_id", "row_type"])
        .reset_index(drop=True)
    )
    _key = event_ledger[["shard", "order_id"]]
    _pairs = _key.drop_duplicates().reset_index(drop=True)
    _pairs["global_order_id"] = _pairs.index + 1
    event_ledger = event_ledger.merge(_pairs, on=["shard", "order_id"], how="left")
    event_ledger["order_id"] = event_ledger["global_order_id"]
    event_ledger = event_ledger.drop(columns=["global_order_id"])
else:
    event_ledger = pd.DataFrame(columns=[
        "row_type", "order_id", "wallet", "condition_id", "outcome",
        "side", "dt", "tx_hash", "price", "order_price",
        "qty_ordered", "qty_filled", "fill_qty",
        "fill_source", "token_winner", "exec_price", "fill_pnl_usdc",
        "wallet_trade_pnl", "shard", "filled_wallet_total_position", "filled_token_total_position", "filled_token_by_wallet_position",
    ])

triggers = event_ledger[event_ledger["row_type"] == "trigger"]
fills = event_ledger[event_ledger["row_type"] == "fill"]
filled_orders = fills["order_id"].nunique()

print(f"Trigger events    : {len(triggers):>7,}")
print(f"Fill rows         : {len(fills):>7,}")
print(f"Orders with fills : {filled_orders:>7,}")
if len(triggers) > 0:
    print(f"Order fill rate   : {filled_orders/len(triggers)*100:.1f}%")
else:
    print("No trigger events found — check WATCHED_WALLETS and the period.")



Trigger events    :  65,181
Fill rows         :  26,282
Orders with fills :  17,220
Order fill rate   : 26.4%


## Summary statistics

In [278]:
fee = FEE_BPS / 10_000.0

if not fills.empty:
    filled_copy_pnl    = fills["fill_pnl_usdc"].sum()
    total_qty_filled   = fills["fill_qty"].sum()
    total_notional     = (fills["fill_qty"] * fills["exec_price"]).sum()
    orders_with_fills  = fills["order_id"].nunique()
    partial_orders     = (fills.groupby("order_id").size() > 1).sum()
    win_rate           = fills.groupby("order_id")["token_winner"].first().mean()
    avg_exec_price     = fills["exec_price"].mean()
    fill_src_counts    = fills["fill_source"].value_counts()

    # Partial-fill statistics
    trig_qty = triggers.set_index("order_id")["qty_ordered"]
    fill_qty = triggers.set_index("order_id")["qty_filled"]
    fill_pct = (fill_qty / trig_qty.clip(lower=1e-12) * 100).replace([float('inf'), float('nan')], 0)

    print(f"=== Copy-strategy performance ===")
    print(f"  Orders triggered    : {len(triggers):>7,}")
    print(f"  Orders with fills   : {orders_with_fills:>7,}  ({orders_with_fills/len(triggers)*100:.1f}%)")
    print(f"  Orders multi-fill   : {partial_orders:>7,}  ({partial_orders/max(orders_with_fills,1)*100:.1f}% of filled)")
    print(f"  Avg fill %          : {fill_pct[fill_pct>0].mean():>7.1f}%")
    print(f"  Total qty filled    : {total_qty_filled:>10,.2f} tokens")
    print(f"  Total notional      : {total_notional:>10,.2f} USDC")
    print(f"  Net PnL (USDC)      : {filled_copy_pnl:>10,.2f}")
    print(f"  Net ROI on notional : {filled_copy_pnl/total_notional*100:>8.2f}%")
    print(f"  Win rate (by order) : {win_rate*100:>8.2f}%")
    print(f"  Avg exec price      : {avg_exec_price:>8.4f}")
    print(f"\n  Fill source breakdown:")
    for src, cnt in fill_src_counts.items():
        print(f"    {src:<20s}: {cnt:>6,}  ({cnt/len(fills)*100:.1f}%)")
else:
    print("No fills — nothing to summarise.")

# Watched-wallet cohort summary
wallet_pnl = triggers["wallet_trade_pnl"].sum()
print(f"\n=== Watched-wallet cohort ===")
print(f"  Total trades        : {len(triggers):>7,}")
print(f"  Total trade PnL     : {wallet_pnl:>10,.2f} USDC")


=== Copy-strategy performance ===
  Orders triggered    :  65,181
  Orders with fills   :  17,220  (26.4%)
  Orders multi-fill   :   5,221  (30.3% of filled)
  Avg fill %          :    88.2%
  Total qty filled    : 215,804.26 tokens
  Total notional      : 110,642.24 USDC
  Net PnL (USDC)      :   3,066.24
  Net ROI on notional :     2.77%
  Win rate (by order) :    57.60%
  Avg exec price      :   0.5489

  Fill source breakdown:
    same_token          : 19,038  (72.4%)
    opposite_token      :  7,244  (27.6%)

=== Watched-wallet cohort ===
  Total trades        :  65,181
  Total trade PnL     : 556,208.11 USDC


## Event ledger preview

In [279]:
event_ledger[
    (event_ledger['row_type'] == 'fill')
    & (event_ledger['condition_id'] == '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20')
    & (event_ledger['wallet'] == '0x0b219cf3d297991b58361dbebdbaa91e56b8deb6')
    ].head(500)

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard


In [280]:
pd.set_option('display.max_rows', 100)

In [281]:
display_cols = [
    "row_type", "order_id", "wallet", "condition_id", "outcome", "side",
    "dt", "tx_hash", "price", "order_price", "exec_price",
    "qty_ordered", "qty_filled", "fill_qty",
    "fill_source", "token_winner", "fill_pnl_usdc", "filled_wallet_total_position", "filled_token_total_position", "filled_token_by_wallet_position", "wallet_trade_pnl",
]
available = [c for c in display_cols if c in event_ledger.columns]

# Show a few trigger+fill pairs
sample_orders = event_ledger["order_id"].unique()[:5]
event_ledger[
    event_ledger["order_id"].isin(sample_orders)
][available].sort_values(["order_id", "row_type"])


,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,fill_pnl_usdc,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,wallet_trade_pnl
0,trigger,1,0x1c144e30f405a25f991cbd8baa15d40599090869,0x0b35eb5df8e17d42cba0909d8bb78e635cb0d014e6fba6322c173cfea6337cd9,No,BUY,2026-03-16 00:00:27+00:00,0x6d88a64f2353e54e5469de1ee4d68b698c0691d84fb19efcf9d763311debb2f1,0.210,0.210,...,4.761905e+01,0.0,NaN,None,False,NaN,0.0,0.0,0.0,-1.260744e+01
1,trigger,2,0xdf409ebc2cda86c98f3abf7666ff4bdc01ff38b8,0x0b35eb5df8e17d42cba0909d8bb78e635cb0d014e6fba6322c173cfea6337cd9,No,BUY,2026-03-16 00:00:27+00:00,0x6d88a64f2353e54e5469de1ee4d68b698c0691d84fb19efcf9d763311debb2f1,0.220,0.220,...,4.545455e+01,0.0,NaN,None,False,NaN,0.0,0.0,0.0,-1.482800e+01
2,trigger,3,0x1c144e30f405a25f991cbd8baa15d40599090869,0x7ba50f79e73433d09a479b62fec39a86f7807fc232e7854e73eccb969e97c149,Yes,BUY,2026-03-16 00:01:29+00:00,0x6a66e304c5d68e165b3c50eb5a262d2f1e3e1a2c41225f55745c451be39fc298,0.048,0.048,...,2.131628e-14,0.0,NaN,None,False,NaN,0.0,0.0,0.0,-1.023182e-15
3,trigger,4,0x87650b9f63563f7c456d9bbcceee5f9faf06ed81,0xbc7438f5fe116c13ac1586748dafb7e659d26a8726a8004570d7654e86f16c3e,Yes,BUY,2026-03-16 00:01:29+00:00,0xc0aac825321efc6f72bfb7606a96b8a659d8a16530cd01d2ace366dc7c9daf6a,0.002,0.002,...,2.146700e+02,0.0,NaN,None,False,NaN,0.0,0.0,0.0,-4.293400e-01
4,trigger,5,0x0ad7f3411bc87f0b5362155e638f04ef05700971,0xbeb4a45b4d634a564c084e5c676cfcc11ef3b4aeeff23c4f4fbd50d39240ddec,No,BUY,2026-03-16 00:02:43+00:00,0x1df81bfcb6bcd72e477ed74f9649245f85e75c075f2ac0e5ae91befdf244a4bb,0.003,0.003,...,2.802000e+01,0.0,NaN,None,False,NaN,0.0,0.0,0.0,-8.406000e-02


In [282]:
event_ledger[event_ledger['side'] == 'BUY'].groupby('wallet').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    buy_count=('side', 'count')
)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,buy_count
wallet,,,
0x002dcd37b0b8fa8db98236e599fe1b90d6272561,135.946958,-4614.885066,489
0x04ec75cbba2a7f2407eb0a48a8468d25ee14580f,-12.404948,-2640.001790,156
0x0ad7f3411bc87f0b5362155e638f04ef05700971,-16.345740,-40.994777,2040
0x0b652d3e55be0e1d50c2f7be39358585c415293e,-40.974356,8233.402549,982
0x0c0e270cf879583d6a0142fc817e05b768d0434e,40.130276,26410.829441,919
0x0cb10c40b0776e9ee8cef970af85724654dda76c,98.452911,11460.951401,1335
0x0dedae6a02ea2ff8018ba5f277632919ed1c9025,4.111735,2251.383105,329
0x0e88a89ce8ab6649c57958d5db329a98864a48fd,2.847423,105.000000,28
0x134a63b764ac7b008356e8db1857db94e6b09e42,2.839006,-217.780089,859


In [283]:
event_ledger[event_ledger['side'] == 'BUY'].groupby('condition_id').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    buy_count=('side', 'count')
).sort_values('fill_pnl_usdc_sum', ascending=False).head(10)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,buy_count
condition_id,,,
0x8183a221622f2d5d2b46da86c80d55a61f43d35829589e03c1efae9d072638c5,672.364263,-75132.971243,1087
0x3f297f4f9083bb1dcd118683d2b75ee79064997cf6ad078a85fd612c3de85083,477.778944,2511.537691,271
0xa9937410f8aae98cc6440a7480ad832be5a1a998afcf5a6e77c6f5aa4c0f259f,449.735025,31935.746988,427
0x6aa39e488afd5dd8e1cd995258c28c3c485dbc4d467d4eb4d94fc6f30e172833,291.033736,4658.337304,352
0x9805dc4c454aca5de1d6f6e287d04a7744099eb7baeab596ceb09c3cee6bc9a8,255.098936,1829.160352,131
0x773abaa5fe55e5cde51a261f444b7921652a4e059ead6b3be9fe56499c2d4609,212.920439,-5225.982733,988
0x959c88f990077233716e9e5bd068f2a457d515000c0fcbe57102154258a2ac74,200.323570,7060.673445,131
0x9dd27666ad05dbe8ac5547e79d634fc306578b60b71f96ab4347f020b46e9413,162.935177,13299.481061,217
0xbc0a2d49b4bea7145a8ca4f43644f82e8013620f45f80665406415c4dfe40966,161.780848,13328.865570,301


In [284]:
(fills[fills['condition_id'] == '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20'][['dt', 'side', 'outcome',
                                                                                                        'price', 'fill_qty', 'exec_price', 
                                                                                                        # 'filled_wallet_total_position',
                                                                                                          'filled_token_total_position', 
                                                                                                          'filled_token_by_wallet_position', 
                                                                                                          'fill_pnl_usdc',
                                                                                                          'order_id','wallet', 'tx_hash']]
 .sort_values('dt')
 .head(10))

,dt,side,outcome,price,fill_qty,exec_price,filled_token_total_position,filled_token_by_wallet_position,fill_pnl_usdc,order_id,wallet,tx_hash


In [285]:
sample_filled_orders = fills["order_id"].unique()[:50]
event_ledger[
    (event_ledger['side'] == 'SELL') &
    (event_ledger["order_id"].isin(sample_filled_orders))
].sort_values(["order_id", "dt"])

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard
28,trigger,21,0xeafc14c8639ef6fbf882b74c3e6de2fe72d10d5c,0x8df5deceb6164ab03aff80ed36e72a916d6bde004288dab11a6fd66aff6e08c0,Yes,SELL,2026-03-16 00:12:25+00:00,0x3f8b031e1ed7fc77c4636e202f4dd75a8a84a44d88ef63232244840ca35dfc99,0.990000,0.990000,...,NaN,None,False,NaN,NaN,19.8000,1.392404,23.431432,1.392404,8
47,fill,21,0xeafc14c8639ef6fbf882b74c3e6de2fe72d10d5c,0x8df5deceb6164ab03aff80ed36e72a916d6bde004288dab11a6fd66aff6e08c0,Yes,SELL,2026-03-16 00:14:07+00:00,0xc128fa8f3db46198055fc2ed2d0331de755a8aec003a486326e965de33f773f1,0.990000,NaN,...,1.010100,same_token,False,0.990000,0.998999,NaN,0.000000,42.881968,0.000000,8
43,trigger,29,0xeafc14c8639ef6fbf882b74c3e6de2fe72d10d5c,0x8df5deceb6164ab03aff80ed36e72a916d6bde004288dab11a6fd66aff6e08c0,Yes,SELL,2026-03-16 00:13:41+00:00,0x002ac6c007a2768ec83908413994c303eb98f8d84beedab7c866e647c588891f,0.970000,0.970000,...,NaN,None,False,NaN,NaN,128.0206,1.392404,44.274372,1.392404,8
45,fill,29,0xeafc14c8639ef6fbf882b74c3e6de2fe72d10d5c,0x8df5deceb6164ab03aff80ed36e72a916d6bde004288dab11a6fd66aff6e08c0,Yes,SELL,2026-03-16 00:13:57+00:00,0x05893ef9c545ed7371b12abee2c784fdab3e34b6e4b25ab3e1f9ecf71ed75f36,0.970000,NaN,...,1.392404,same_token,False,0.970000,1.349281,NaN,0.000000,42.881968,0.000000,8
59,trigger,41,0xbff705be325e15f469da68aa3c0aa43306cf554f,0xb688396158ca4b9d83596401bcdbd75fd4fcefea566fe094a0333390dbfdb308,Yes,SELL,2026-03-16 00:22:09+00:00,0xaa304b7c08b5af80f7946bf6d9c238146d9a0049f72675bb58988f64f47f64cd,0.510000,0.510000,...,NaN,None,False,NaN,NaN,2.5500,5.000000,5.000000,5.000000,b
60,fill,41,0xbff705be325e15f469da68aa3c0aa43306cf554f,0xb688396158ca4b9d83596401bcdbd75fd4fcefea566fe094a0333390dbfdb308,Yes,SELL,2026-03-16 00:22:43+00:00,0xf94c3cd6d3b88198ec464e6b06b295342b509a59474bbdc812ca42f70b50e32a,0.510000,NaN,...,5.000000,same_token,False,0.510000,2.547450,NaN,0.000000,0.000000,0.000000,b
73,trigger,48,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0x8df5deceb6164ab03aff80ed36e72a916d6bde004288dab11a6fd66aff6e08c0,Yes,SELL,2026-03-16 00:37:11+00:00,0x39b87b6173dcb7b5b9effa70d2f49b878ff257291f42f0456b800c5b9eda76ac,0.940181,0.940181,...,NaN,None,False,NaN,NaN,615.4237,11.844328,53.520266,11.844328,8
74,fill,48,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0x8df5deceb6164ab03aff80ed36e72a916d6bde004288dab11a6fd66aff6e08c0,Yes,SELL,2026-03-16 00:37:21+00:00,0x74b74732cd3f80dd75e7024c5fcae86a3efa9dff713ad19675e6bb9868763cca,0.941000,NaN,...,5.000000,opposite_token,False,0.940181,4.696204,NaN,6.844328,48.520266,6.844328,8
94,trigger,62,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,0x399c6423703b85982280a253f995475f5129f41c6dfc75d646920a4f02f60724,No,SELL,2026-03-16 00:56:11+00:00,0x214c700c9fe6364c297cce002ae71e03301a792bb86d4f40045d0e56bc8b3158,0.003000,0.003000,...,NaN,None,False,NaN,NaN,0.1620,100.000000,100.000000,100.000000,3
95,fill,62,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,0x399c6423703b85982280a253f995475f5129f41c6dfc75d646920a4f02f60724,No,SELL,2026-03-16 00:56:41+00:00,0xff841f646924fd4e9aee7e45f626e8bc6c808c9174c79357c12294bdda05c837,0.005000,NaN,...,10.030090,opposite_token,False,0.003000,0.030060,NaN,89.969910,89.969910,89.969910,3


## Build PnL time series

In [286]:
def resample_hourly_series(df: pd.DataFrame, dt_col: str, pnl_col: str) -> pd.DataFrame:
    """Resample a PnL column to 1-h buckets and compute cumulative sum."""
    if df.empty or pnl_col not in df.columns:
        return pd.DataFrame(columns=["trade_dt", "net_pnl_usdc", "cum_net_pnl_usdc"])
    hourly = (
        df.assign(trade_dt=df[dt_col].dt.floor("1h"))
        .groupby("trade_dt", as_index=False)[pnl_col]
        .sum()
        .rename(columns={pnl_col: "net_pnl_usdc"})
        .sort_values("trade_dt")
        .reset_index(drop=True)
    )
    hourly["cum_net_pnl_usdc"] = hourly["net_pnl_usdc"].cumsum()
    return hourly


def with_zero_anchor(hourly: pd.DataFrame) -> pd.DataFrame:
    """Prepend a zero anchor one hour before the first bucket."""
    if hourly.empty:
        return hourly
    anchor = pd.DataFrame({
        "trade_dt": [hourly["trade_dt"].min() - pd.Timedelta(hours=1)],
        "net_pnl_usdc": [0.0],
        "cum_net_pnl_usdc": [0.0],
    })
    return pd.concat([anchor, hourly], ignore_index=True)


# Wallet cohort PnL: from trigger rows (their actual trade_pnl)
wallet_hourly = resample_hourly_series(triggers, "dt", "wallet_trade_pnl")

# Copy-strategy PnL: from fill rows
copy_hourly = resample_hourly_series(fills, "dt", "fill_pnl_usdc")

print(f"Wallet cohort hourly buckets : {len(wallet_hourly)}")
print(f"Copy strategy hourly buckets : {len(copy_hourly)}")

Wallet cohort hourly buckets : 1224
Copy strategy hourly buckets : 1155


In [287]:
wallet_hourly.head()

,trade_dt,net_pnl_usdc,cum_net_pnl_usdc
0,2026-03-16 00:00:00+00:00,-795.981874,-795.981874
1,2026-03-16 01:00:00+00:00,9.648464,-786.333410
2,2026-03-16 02:00:00+00:00,-433.057535,-1219.390945
3,2026-03-16 03:00:00+00:00,-795.914919,-2015.305864
4,2026-03-16 04:00:00+00:00,-730.392700,-2745.698564


## Cumulative PnL chart

In [288]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = go.Figure()

# ── Wallet cohort trace ───────────────────────────────────────────────────────
if not wallet_hourly.empty:
    wh = with_zero_anchor(wallet_hourly)
    fig.add_trace(go.Scatter(
        x=wh["trade_dt"],
        y=wh["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="dot", width=2),
        opacity=0.7,
        name="Watched wallets (raw PnL)",
        hovertemplate=(
            "<b>Watched wallets</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

# ── Copy-strategy trace ───────────────────────────────────────────────────────
if not copy_hourly.empty:
    ch = with_zero_anchor(copy_hourly)
    fig.add_trace(go.Scatter(
        x=ch["trade_dt"],
        y=ch["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="solid", width=2),
        name="Copy strategy (filled)",
        hovertemplate=(
            "<b>Copy strategy</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

fig.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5)

fig.update_layout(
    template="plotly_white",
    height=480,
    title=dict(
        text=(
            f"Copy-trade backtest — cumulative PnL (1h) | "
            f"{period_start} → {period_end} | "
            f"{len(WATCHED_WALLETS)} wallets | "
            f"worse_price={WORSE_PRICE_OFFSET:.2f} | "
            f"horizon={int(FILL_HORIZON_SECONDS)}s"
        ),
        font=dict(size=13),
    ),
    xaxis_title="Time (1h buckets)",
    yaxis_title="Cumulative net PnL (USDC)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

fig.show()

## Diagnostics

In [289]:
# Fill rate by side (order-level)
if not triggers.empty:
    trig_by_side = triggers.groupby("side").size().rename("triggers")
    fill_by_side = fills.groupby("side")["order_id"].nunique().rename("orders_filled")
    fill_rows_by_side = fills.groupby("side").size().rename("fill_rows")
    side_summary = pd.concat([trig_by_side, fill_by_side, fill_rows_by_side], axis=1).fillna(0).astype(int)
    side_summary["fill_rate"] = (side_summary["orders_filled"] / side_summary["triggers"] * 100).round(1)
    display(side_summary)


,triggers,orders_filled,fill_rows,fill_rate
side,,,,
BUY,56007,11321,17360,20.2
SELL,9174,5899,8922,64.3


In [290]:
# Fill source breakdown by side
if not fills.empty:
    display(
        fills.groupby(["side", "fill_source"]).size()
        .rename("count")
        .reset_index()
        .sort_values(["side", "fill_source"])
    )

,side,fill_source,count
0,BUY,opposite_token,4323
1,BUY,same_token,13037
2,SELL,opposite_token,2921
3,SELL,same_token,6001


In [291]:
df = event_ledger.assign(
    is_trigger=event_ledger["row_type"].eq("trigger"),
    is_fill=event_ledger["row_type"].eq("fill"),
)

# --- wallet-level stats (row-based) ---
wallet_stats = (
    df.groupby("wallet")
    .agg(
        total_triggers=("is_trigger", "sum"),
        total_fills=("is_fill", "sum"),
        total_fill_pnl=("fill_pnl_usdc", "sum"),
        total_trade_pnl=("wallet_trade_pnl", "sum"),
        total_pnl=("fill_pnl_usdc", "sum"),
    )
)

# --- order-level logic (trigger -> fill) ---
order_stats = (
    df.groupby(["wallet", "order_id"])
    .agg(
        has_trigger=("is_trigger", "any"),
        has_fill=("is_fill", "any"),
    )
    .assign(trigger_with_fill=lambda x: x["has_trigger"] & x["has_fill"])
    .groupby("wallet")
    .agg(triggers_with_fill=("trigger_with_fill", "sum"))
)

# --- combine ---
result = wallet_stats.join(order_stats)

# --- derived metric ---
result["fill_ratio"] = (
    result["triggers_with_fill"] / result["total_triggers"]
).fillna(0)

result.sort_values("total_pnl", ascending=False).head()

,total_triggers,total_fills,total_fill_pnl,total_trade_pnl,total_pnl,triggers_with_fill,fill_ratio
wallet,,,,,,,
0x88c4919de76e526d55a32c1f8afb439dd1f1129a,710,501,337.889722,74903.971850,337.889722,263,0.370423
0xffb0b9b292e406fd250854a35a0c9bd5612afa37,756,486,302.965613,7751.147388,302.965613,269,0.355820
0xa0926fbc4a90e29e479b459ef92a147ebe5b11e3,848,247,267.477718,-22311.416001,267.477718,156,0.183962
0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007,1515,1190,265.185680,2911.565894,265.185680,750,0.495050
0xd44e974a3edb232aa4aedbdcc59792b76a5f67e2,1072,498,263.845138,43005.507414,263.845138,283,0.263993


In [292]:
len(df)

91463

In [293]:
# get diff between trigger and fill timestamp per order_id
# without lambda, using groupby + agg + merge
trigger_df = df[df['row_type'] == 'trigger'].groupby('order_id')['dt'].min().reset_index()
fill_df = df[df['row_type'] == 'fill'].groupby('order_id')['dt'].min().reset_index()

merged_df = pd.merge(trigger_df, fill_df, on='order_id', how='outer', suffixes=('_trigger', '_fill'))
merged_df['diff_dt'] = merged_df['dt_fill'] - merged_df['dt_trigger']

In [294]:
merged_df[merged_df['diff_dt'].notnull()]['diff_dt'].mean().total_seconds()

57.857084

In [295]:
result[result['total_fill_pnl'] < 0].sort_values("total_fill_pnl").index.tolist()

['0x5739ddf8672627ce076eff5f444610a250075f1a',
 '0xc81c1df1f183e1b137119e590e9d7b64e805e454',
 '0xc2073845d851369fbb470d60c5137e9abe0ab24e',
 '0xbff705be325e15f469da68aa3c0aa43306cf554f',
 '0xdfe3fedc5c7679be42c3d393e99d4b55247b73c4',
 '0x695c1dbc0cf3d4758c8879ea5139c1615d9310a1',
 '0xe6266639b0f2fe7019b5a04f02f60d8a13276771',
 '0x40c2348351024cce5ba6fccd6015b77bbb77b2fc',
 '0x7656ed7f597a0a61cd307591db198a42b2a7194b',
 '0xcf6a714618a328c608a1c70cb62a31a6bef3f9d0',
 '0xcc3e87226912d35893b678eaa8a39d57b6fe37ef',
 '0x6bc74c392c320cfe10d5be61db978a58c8444ad4',
 '0x0b652d3e55be0e1d50c2f7be39358585c415293e',
 '0xd5555147366b7e5bfcea444f4a06a745c18be28a',
 '0x9ccc0ca985cd638f106d5f355cc2e4d11d4ce7e8',
 '0x134a63b764ac7b008356e8db1857db94e6b09e42',
 '0x5c08f27559956e9c4fa008c65f79017fe548eba9',
 '0x04ec75cbba2a7f2407eb0a48a8468d25ee14580f',
 '0x9dd9e18b9dcc393705e73966bc79eccbbd9108bd',
 '0x5446fe5868e4e6da0ab784b005fe423df62218f6',
 '0xf658449199d0bcf544de0ff928a3b66685f3dcfe',
 '0x8b72cb885

In [296]:
# Per-wallet PnL breakdown
if not fills.empty:
    wallet_pnl_df = (
        fills.groupby("wallet")
        .agg(
            filled_orders=("order_id", "nunique"),
            fill_rows=("order_id", "count"),
            net_pnl_usdc=("fill_pnl_usdc", "sum"),
            total_qty=("fill_qty", "sum"),
            win_rate=("token_winner", lambda s: fills.loc[s.index].groupby("order_id")["token_winner"].first().mean()),
        )
        .assign(win_rate=lambda d: (d["win_rate"] * 100).round(1))
        .sort_values("net_pnl_usdc", ascending=False)
        .reset_index()
    )
    display(wallet_pnl_df)


,wallet,filled_orders,fill_rows,net_pnl_usdc,total_qty,win_rate
0,0x88c4919de76e526d55a32c1f8afb439dd1f1129a,263,501,337.889722,5027.537986,90.5
1,0xffb0b9b292e406fd250854a35a0c9bd5612afa37,269,486,302.965613,3313.531338,82.9
2,0xa0926fbc4a90e29e479b459ef92a147ebe5b11e3,156,247,267.477718,2000.697930,84.6
3,0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007,750,1190,265.185680,9605.123486,70.4
4,0xd44e974a3edb232aa4aedbdcc59792b76a5f67e2,283,498,263.845138,3218.426630,74.6
5,0x41583f2efc720b8e2682750fffb67f2806fece9f,390,717,233.239945,6064.065018,32.3
6,0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292,325,510,205.415485,4192.943226,54.2
7,0x6560abb717a8e5f3afaacd1a31e9c03eed71e164,400,582,201.070146,5187.579670,69.2
8,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,254,440,194.473844,2742.067694,64.6
9,0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62,185,319,183.453351,2749.080927,71.4


In [297]:
wallet_pnl_df.head(10)['wallet'].tolist()

['0x88c4919de76e526d55a32c1f8afb439dd1f1129a',
 '0xffb0b9b292e406fd250854a35a0c9bd5612afa37',
 '0xa0926fbc4a90e29e479b459ef92a147ebe5b11e3',
 '0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007',
 '0xd44e974a3edb232aa4aedbdcc59792b76a5f67e2',
 '0x41583f2efc720b8e2682750fffb67f2806fece9f',
 '0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292',
 '0x6560abb717a8e5f3afaacd1a31e9c03eed71e164',
 '0x66c1a6fe836ff555ca32848646acedbbe93bfa3f',
 '0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62']

In [298]:
# Orders that were NOT filled at all
filled_order_ids = set(fills["order_id"].unique()) if not fills.empty else set()
unfilled_triggers = triggers[~triggers["order_id"].isin(filled_order_ids)]
print(f"Unfilled orders    : {len(unfilled_triggers):,} ({len(unfilled_triggers)/max(len(triggers),1)*100:.1f}% of all triggers)")

# Partially filled orders (filled but qty_filled < qty_ordered)
if not fills.empty:
    filled_trig = triggers[triggers["order_id"].isin(filled_order_ids)].copy()
    partial_mask = filled_trig["qty_filled"] < filled_trig["qty_ordered"] - 1e-9
    partial_fills = filled_trig[partial_mask]
    print(f"Partially filled   : {len(partial_fills):,} ({len(partial_fills)/max(len(filled_trig),1)*100:.1f}% of filled orders)")
    print(f"Fully filled       : {len(filled_trig)-len(partial_fills):,}")

if not unfilled_triggers.empty:
    print("\nUnfilled breakdown by side:")
    display(unfilled_triggers.groupby("side").size().rename("count").reset_index())


Unfilled orders    : 47,961 (73.6% of all triggers)
Partially filled   : 3,607 (20.9% of filled orders)
Fully filled       : 13,613

Unfilled breakdown by side:


,side,count
0,BUY,44686
1,SELL,3275


In [299]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
fills[fills['order_id'] == 34]

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard
51,fill,34,0x1c144e30f405a25f991cbd8baa15d40599090869,0x8df5deceb6164ab03aff80ed36e72a916d6bde004288dab11a6fd66aff6e08c0,Yes,BUY,2026-03-16 00:16:49+00:00,0x5dd2fd093806ecccf6c2aef624aea4738fb0cf03698f8a670ca5c1b8d36c4dcd,0.913,NaN,...,10.638298,opposite_token,False,0.94,-10.01,NaN,41.675938,53.520266,41.675938,8


In [300]:
fills.groupby("wallet")["fill_pnl_usdc"].sum().sort_values(ascending=False, key=abs).head(100)

wallet
0x88c4919de76e526d55a32c1f8afb439dd1f1129a    337.889722
0xffb0b9b292e406fd250854a35a0c9bd5612afa37    302.965613
0xa0926fbc4a90e29e479b459ef92a147ebe5b11e3    267.477718
0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007    265.185680
0xd44e974a3edb232aa4aedbdcc59792b76a5f67e2    263.845138
0x41583f2efc720b8e2682750fffb67f2806fece9f    233.239945
0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292    205.415485
0x6560abb717a8e5f3afaacd1a31e9c03eed71e164    201.070146
0x66c1a6fe836ff555ca32848646acedbbe93bfa3f    194.473844
0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62    183.453351
0x5739ddf8672627ce076eff5f444610a250075f1a   -182.209348
0x9238743eeba8bbfe9e85ac7ba2e1e3d77877b73e    177.237249
0x1d1aa3439835cc8dbb5e55d615bee3f39b0d324e    175.537697
0xc81c1df1f183e1b137119e590e9d7b64e805e454   -169.006265
0x43440ab002eaac9fede6f9d21bea96d84228f90d    162.325129
0xc2073845d851369fbb470d60c5137e9abe0ab24e   -156.604489
0x92d0cb81e6c891b835c8e0272e8c323095bd269e    155.253705
0x83c56c56bc67c0b99b38ab

In [301]:
trigger_wallets = triggers[["order_id", "wallet"]].set_index("order_id")["wallet"]

fills_with_wallet = fills.merge(trigger_wallets, on="order_id", how="left", suffixes=("", "_trigger"))

fills_with_wallet['notional'] = fills_with_wallet['fill_qty'] * fills_with_wallet['exec_price']

fills_with_wallet.groupby(["wallet", "condition_id"]).agg(
    pnl=("fill_pnl_usdc", "sum"),
    orders_filled=("order_id", "nunique")
    ).sort_values(['wallet', "pnl"], ascending=[True, False]).head(1)

,,pnl,orders_filled
wallet,condition_id,,
0x002dcd37b0b8fa8db98236e599fe1b90d6272561,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,44.730585,8


In [302]:
fills_with_wallet.head()

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard,wallet_trigger,notional
0,fill,10,0x87650b9f63563f7c456d9bbcceee5f9faf06ed81,0x217f15d80517f21e63be8b382e46e6404a765bd106021b8f1e52573a1fbd7f57,Yes,BUY,2026-03-16 00:11:45+00:00,0x983f39fb32d7bf03f9537e9a413de362fac4e467170b6e53dbcac9d7c0c00ad9,0.350,NaN,...,True,0.350000,18.561429,NaN,28.571429,28.571429,28.571429,2,0x87650b9f63563f7c456d9bbcceee5f9faf06ed81,10.000000
1,fill,11,0x87650b9f63563f7c456d9bbcceee5f9faf06ed81,0x217f15d80517f21e63be8b382e46e6404a765bd106021b8f1e52573a1fbd7f57,Yes,BUY,2026-03-16 00:12:03+00:00,0x164c6a4cec552c98ecf51dcbc2abcb8e3dba4c8c012b32937567e51478a41311,0.220,NaN,...,True,0.350000,18.274654,NaN,56.701429,56.701429,56.701429,2,0x87650b9f63563f7c456d9bbcceee5f9faf06ed81,9.845500
2,fill,18,0x87650b9f63563f7c456d9bbcceee5f9faf06ed81,0xea7bbe9dd8364642fc646aa4a86dd6fc309f60e4ccd451a4d4a473aa6c8f7730,Yes,BUY,2026-03-16 00:12:09+00:00,0x59940733dd9989b10eea4fc267443a171b3f7defcb68eef0583570cfce21484f,0.315,NaN,...,True,0.317000,10.240245,NaN,15.000000,15.000000,15.000000,e,0x87650b9f63563f7c456d9bbcceee5f9faf06ed81,4.755000
3,fill,14,0xeafc14c8639ef6fbf882b74c3e6de2fe72d10d5c,0x8df5deceb6164ab03aff80ed36e72a916d6bde004288dab11a6fd66aff6e08c0,Yes,BUY,2026-03-16 00:12:25+00:00,0x94b8cbcdf11453ef96862f15368239e03bfe00d70466aff410529cfccbe926c2,0.210,NaN,...,False,0.210000,-0.292697,NaN,1.392404,21.587104,1.392404,8,0xeafc14c8639ef6fbf882b74c3e6de2fe72d10d5c,0.292405
4,fill,16,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0x8df5deceb6164ab03aff80ed36e72a916d6bde004288dab11a6fd66aff6e08c0,Yes,BUY,2026-03-16 00:12:25+00:00,0x94b8cbcdf11453ef96862f15368239e03bfe00d70466aff410529cfccbe926c2,0.840,NaN,...,False,0.844286,-4.225651,NaN,5.000000,15.194700,5.000000,8,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,4.221430


In [303]:
wallet_pnls = (
    fills_with_wallet
    .groupby(["wallet", "condition_id"])
    .agg(
        pnl=("fill_pnl_usdc", "sum"),
        orders_filled=("order_id", "nunique"),
        notional=("notional", "sum"),
    )
    .groupby("wallet")
    .agg(
        pnl=("pnl", "sum"),
        notional=("notional", "sum"),
        unique_conditions=("pnl", "size"),  # ✅ count rows
        total_orders_filled=("orders_filled", "sum"),
    )
    .sort_values("pnl", ascending=False)
)

In [304]:
wallet_pnls.head()

,pnl,notional,unique_conditions,total_orders_filled
wallet,,,,
0x88c4919de76e526d55a32c1f8afb439dd1f1129a,337.889722,3764.824803,41,263
0xffb0b9b292e406fd250854a35a0c9bd5612afa37,302.965613,2243.864066,69,269
0xa0926fbc4a90e29e479b459ef92a147ebe5b11e3,267.477718,1464.151562,30,156
0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007,265.185680,6457.395787,68,750
0xd44e974a3edb232aa4aedbdcc59792b76a5f67e2,263.845138,1675.821476,33,283


In [305]:
MAX_WALLET_EXPOSURE = 1000.0  # USDC
wallet_pnls['pnl_limited'] = np.where(
    wallet_pnls['notional'] <= MAX_WALLET_EXPOSURE,
    wallet_pnls['pnl'],
    wallet_pnls['pnl'] * MAX_WALLET_EXPOSURE / wallet_pnls['notional']
)

In [306]:
wallet_pnls.head(10)

,pnl,notional,unique_conditions,total_orders_filled,pnl_limited
wallet,,,,,
0x88c4919de76e526d55a32c1f8afb439dd1f1129a,337.889722,3764.824803,41,263,89.749123
0xffb0b9b292e406fd250854a35a0c9bd5612afa37,302.965613,2243.864066,69,269,135.019593
0xa0926fbc4a90e29e479b459ef92a147ebe5b11e3,267.477718,1464.151562,30,156,182.684447
0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007,265.185680,6457.395787,68,750,41.066970
0xd44e974a3edb232aa4aedbdcc59792b76a5f67e2,263.845138,1675.821476,33,283,157.442270
0x41583f2efc720b8e2682750fffb67f2806fece9f,233.239945,2425.447113,62,390,96.163690
0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292,205.415485,2333.026250,23,325,88.046796
0x6560abb717a8e5f3afaacd1a31e9c03eed71e164,201.070146,2708.245639,52,400,74.243689
0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,194.473844,1624.048188,92,254,119.746351


In [307]:
result = (
    fills_with_wallet
    # 1️⃣ PnL per market (condition) per wallet
    .groupby(["wallet", "condition_id"])["fill_pnl_usdc"]
    .sum()
    
    # 2️⃣ Then per wallet
    .groupby("wallet")
    .agg(
        unique_conditions="count",   # number of markets traded
        median_market_pnl="median",  # median PnL across markets
    )
    .sort_values("unique_conditions", ascending=False)
    .head(20)
)

In [308]:
result.sort_values("median_market_pnl", ascending=False).head(100)

,unique_conditions,median_market_pnl
wallet,,
0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007,68,1.418460
0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,92,0.846818
0xffb0b9b292e406fd250854a35a0c9bd5612afa37,69,0.707433
0x9b2fcb79bdb8a6a6174da327e9335f7889c6d8f8,69,0.598557
0x22dbd51e1a4e10fff072fa24300238c64033190f,115,0.470669
0xdf409ebc2cda86c98f3abf7666ff4bdc01ff38b8,91,0.395400
0x83c56c56bc67c0b99b38abd1c0a3c1aca7a7e193,86,0.298034
0x1d1aa3439835cc8dbb5e55d615bee3f39b0d324e,77,0.132761
0x1c144e30f405a25f991cbd8baa15d40599090869,71,0.106419


In [309]:
fills_with_wallet.head()

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard,wallet_trigger,notional
0,fill,10,0x87650b9f63563f7c456d9bbcceee5f9faf06ed81,0x217f15d80517f21e63be8b382e46e6404a765bd106021b8f1e52573a1fbd7f57,Yes,BUY,2026-03-16 00:11:45+00:00,0x983f39fb32d7bf03f9537e9a413de362fac4e467170b6e53dbcac9d7c0c00ad9,0.350,NaN,...,True,0.350000,18.561429,NaN,28.571429,28.571429,28.571429,2,0x87650b9f63563f7c456d9bbcceee5f9faf06ed81,10.000000
1,fill,11,0x87650b9f63563f7c456d9bbcceee5f9faf06ed81,0x217f15d80517f21e63be8b382e46e6404a765bd106021b8f1e52573a1fbd7f57,Yes,BUY,2026-03-16 00:12:03+00:00,0x164c6a4cec552c98ecf51dcbc2abcb8e3dba4c8c012b32937567e51478a41311,0.220,NaN,...,True,0.350000,18.274654,NaN,56.701429,56.701429,56.701429,2,0x87650b9f63563f7c456d9bbcceee5f9faf06ed81,9.845500
2,fill,18,0x87650b9f63563f7c456d9bbcceee5f9faf06ed81,0xea7bbe9dd8364642fc646aa4a86dd6fc309f60e4ccd451a4d4a473aa6c8f7730,Yes,BUY,2026-03-16 00:12:09+00:00,0x59940733dd9989b10eea4fc267443a171b3f7defcb68eef0583570cfce21484f,0.315,NaN,...,True,0.317000,10.240245,NaN,15.000000,15.000000,15.000000,e,0x87650b9f63563f7c456d9bbcceee5f9faf06ed81,4.755000
3,fill,14,0xeafc14c8639ef6fbf882b74c3e6de2fe72d10d5c,0x8df5deceb6164ab03aff80ed36e72a916d6bde004288dab11a6fd66aff6e08c0,Yes,BUY,2026-03-16 00:12:25+00:00,0x94b8cbcdf11453ef96862f15368239e03bfe00d70466aff410529cfccbe926c2,0.210,NaN,...,False,0.210000,-0.292697,NaN,1.392404,21.587104,1.392404,8,0xeafc14c8639ef6fbf882b74c3e6de2fe72d10d5c,0.292405
4,fill,16,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0x8df5deceb6164ab03aff80ed36e72a916d6bde004288dab11a6fd66aff6e08c0,Yes,BUY,2026-03-16 00:12:25+00:00,0x94b8cbcdf11453ef96862f15368239e03bfe00d70466aff410529cfccbe926c2,0.840,NaN,...,False,0.844286,-4.225651,NaN,5.000000,15.194700,5.000000,8,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,4.221430


In [310]:
from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()
mdf = mdf[
    ~(mdf['primary_tag'].isin(['Sports', 'Crypto']))
    & (mdf['winner_token_id'].notna())
    ].copy().reset_index(drop=True)


In [311]:
mdf.head()

,condition_id,end_date_iso,question,tags,primary_tag,winner_token_id,outcome
0,0x055176c9ebd8775c281ad540d5c16b3323e316507aef2d45c84f061cbc6bbcdc,2022-08-21T00:00:00Z,"MLB: Who will win Boston Red Sox v. Baltimore Orioles, scheduled for August 21, 7:10 PM ET?",[All],All,9612890763764062692282935414227141810568206972440321500296202304471805951204,Orioles
1,0x1409177eae7f1b0406bc0eea9d2715c9f76d88ec7765aed03cf39d59f36008f7,2023-01-06T00:00:00Z,Will a House Speaker be elected by end of Thursday?,[All],All,56419231299380534710656783098291368308638365867243518354348835265264811381897,No
2,0xd6165c1a30269d0799b27ee6ea90c048dfd6b27276cbd2aa7e18fe1c4562faa8,2023-01-14T00:00:00Z,Will Benjamin Netanyahu be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,115040502429502340199300991115762950152511906206476760034830683863926486402417,No
3,0x9da76626846b7a0f7c5388feb280e42a67564ea104123020fbf59a641b179623,2023-01-14T00:00:00Z,Will Xi Jinping be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,55070293174349524004776076543655661478039361630374630352242580072992541740735,No
4,0xe7d21325e5cfccbdaddf6ab6ef9bf477cf3aa6615dd36c8c408da99a1dd83237,2023-01-14T00:00:00Z,Will Volodymyr Zelenskyy be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,5775128592397269721112080310923554806977502333079078864830955469822139503940,No


## Browser plots: PnL and positions

Interactive Plotly charts for cumulative PnL and signed positions. Market hovers include `end_date_iso` from `mdf`.


In [312]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "browser"

TOP_WALLETS_PNL = 10
TOP_MARKETS_PNL = 10
TOP_WALLETS_POSITION = 10
TOP_MARKETS_POSITION = 10

def _short_wallet(wallet: str) -> str:
    wallet = str(wallet)
    return f"{wallet[:6]}...{wallet[-4:]}" if len(wallet) > 14 else wallet

def _short_text(text: str, width: int = 72) -> str:
    if pd.isna(text):
        return "Unknown market"
    text = " ".join(str(text).split())
    return text if len(text) <= width else text[: width - 3] + "..."

def _build_total_ts(df: pd.DataFrame, value_col: str, net_col: str, cum_col: str) -> pd.DataFrame:
    total = (
        df.sort_values("dt")
        .groupby("dt", as_index=False)[value_col]
        .sum()
        .rename(columns={value_col: net_col})
    )
    total[cum_col] = total[net_col].cumsum()
    return total

def _build_position_snapshots(df: pd.DataFrame) -> pd.DataFrame:
    series_cols = [
        "wallet",
        "wallet_label",
        "condition_id",
        "market_label",
        "end_date_iso",
        "market_close_dt",
        "outcome",
    ]

    active = df[
        df["market_close_dt"].isna() | (df["dt"] < df["market_close_dt"])
    ].copy()

    snapshots = (
        active.sort_values(series_cols + ["dt", "order_id"])
        .groupby(series_cols + ["dt"], as_index=False, dropna=False)
        .agg(
            token_position=("filled_token_by_wallet_position", "last"),
            exec_price=("exec_price", "last"),
        )
    )
    snapshots["usdc_position"] = snapshots["token_position"] * snapshots["exec_price"]

    if snapshots.empty:
        snapshots["token_delta"] = pd.Series(dtype=float)
        snapshots["usdc_delta"] = pd.Series(dtype=float)
        return snapshots

    reset = (
        snapshots.groupby(series_cols, as_index=False, dropna=False)
        .agg(
            token_position=("token_position", "last"),
            usdc_position=("usdc_position", "last"),
        )
    )
    reset = reset[
        reset["market_close_dt"].notna()
        & ((reset["token_position"].abs() > 1e-12) | (reset["usdc_position"].abs() > 1e-12))
    ].copy()
    reset["dt"] = reset["market_close_dt"]
    reset["token_position"] = 0.0
    reset["usdc_position"] = 0.0

    combined = pd.concat(
        [
            snapshots[series_cols + ["dt", "token_position", "usdc_position"]],
            reset[series_cols + ["dt", "token_position", "usdc_position"]],
        ],
        ignore_index=True,
    )
    combined = (
        combined.sort_values(series_cols + ["dt"])
        .drop_duplicates(subset=series_cols + ["dt"], keep="last")
        .reset_index(drop=True)
    )
    combined["token_delta"] = combined.groupby(series_cols, dropna=False)["token_position"].diff()
    combined["usdc_delta"] = combined.groupby(series_cols, dropna=False)["usdc_position"].diff()
    combined["token_delta"] = combined["token_delta"].fillna(combined["token_position"])
    combined["usdc_delta"] = combined["usdc_delta"].fillna(combined["usdc_position"])
    return combined

market_meta = (
    mdf[["condition_id", "question", "end_date_iso", "primary_tag"]]
    .drop_duplicates(subset=["condition_id"])
    .copy()
)
market_meta["end_date"] = pd.to_datetime(market_meta["end_date_iso"], utc=True, errors="coerce")
market_meta["market_close_dt"] = market_meta["end_date"].dt.floor("D") + pd.Timedelta(days=1)

plot_fills = fills.copy()
plot_fills["dt"] = pd.to_datetime(plot_fills["dt"], utc=True)
plot_fills = plot_fills.merge(market_meta, on="condition_id", how="left")
plot_fills["wallet_label"] = plot_fills["wallet"].map(_short_wallet)
plot_fills["market_label"] = plot_fills["question"].map(_short_text)
plot_fills["market_label"] = np.where(
    plot_fills["market_label"].eq("Unknown market"),
    plot_fills["condition_id"].str[:12],
    plot_fills["market_label"],
)

pnl_total_ts = _build_total_ts(plot_fills, "fill_pnl_usdc", "net_pnl_usdc", "cum_pnl_usdc")

wallet_pnl_totals = (
    plot_fills.groupby(["wallet", "wallet_label"], as_index=False, dropna=False)
    .agg(total_pnl_usdc=("fill_pnl_usdc", "sum"))
)
top_wallets_pnl = wallet_pnl_totals.reindex(
    wallet_pnl_totals["total_pnl_usdc"].abs().sort_values(ascending=False).index
).head(TOP_WALLETS_PNL).copy()
wallet_pnl_ts = (
    plot_fills[plot_fills["wallet"].isin(top_wallets_pnl["wallet"])]
    .groupby(["wallet", "wallet_label", "dt"], as_index=False, dropna=False)
    .agg(net_pnl_usdc=("fill_pnl_usdc", "sum"))
    .sort_values(["wallet", "dt"])
)
wallet_pnl_ts["cum_pnl_usdc"] = wallet_pnl_ts.groupby("wallet", dropna=False)["net_pnl_usdc"].cumsum()

market_pnl_totals = (
    plot_fills.groupby(["condition_id", "market_label", "end_date_iso"], as_index=False, dropna=False)
    .agg(total_pnl_usdc=("fill_pnl_usdc", "sum"))
)
top_markets_pnl = market_pnl_totals.reindex(
    market_pnl_totals["total_pnl_usdc"].abs().sort_values(ascending=False).index
).head(TOP_MARKETS_PNL).copy()
market_pnl_ts = (
    plot_fills[plot_fills["condition_id"].isin(top_markets_pnl["condition_id"])]
    .groupby(["condition_id", "market_label", "end_date_iso", "dt"], as_index=False, dropna=False)
    .agg(net_pnl_usdc=("fill_pnl_usdc", "sum"))
    .sort_values(["condition_id", "dt"])
)
market_pnl_ts["cum_pnl_usdc"] = market_pnl_ts.groupby("condition_id", dropna=False)["net_pnl_usdc"].cumsum()

granular_position_ts = _build_position_snapshots(plot_fills)

wallet_position_all = (
    granular_position_ts.groupby(["wallet", "wallet_label", "dt"], as_index=False, dropna=False)
    .agg(
        net_token_position=("token_delta", "sum"),
        net_usdc_position=("usdc_delta", "sum"),
    )
    .sort_values(["wallet", "dt"])
)
wallet_position_all["cum_token_position"] = wallet_position_all.groupby("wallet", dropna=False)["net_token_position"].cumsum()
wallet_position_all["cum_usdc_position"] = wallet_position_all.groupby("wallet", dropna=False)["net_usdc_position"].cumsum()
wallet_position_rank = (
    wallet_position_all.groupby(["wallet", "wallet_label"], as_index=False, dropna=False)
    .agg(max_abs_usdc_position=("cum_usdc_position", lambda s: s.abs().max()))
)
top_wallets_position = wallet_position_rank.sort_values("max_abs_usdc_position", ascending=False).head(TOP_WALLETS_POSITION).copy()
wallet_position_ts = wallet_position_all[wallet_position_all["wallet"].isin(top_wallets_position["wallet"])].copy()

market_position_all = (
    granular_position_ts.groupby(["condition_id", "market_label", "end_date_iso", "dt"], as_index=False, dropna=False)
    .agg(
        net_token_position=("token_delta", "sum"),
        net_usdc_position=("usdc_delta", "sum"),
    )
    .sort_values(["condition_id", "dt"])
)
market_position_all["cum_token_position"] = market_position_all.groupby("condition_id", dropna=False)["net_token_position"].cumsum()
market_position_all["cum_usdc_position"] = market_position_all.groupby("condition_id", dropna=False)["net_usdc_position"].cumsum()
market_position_rank = (
    market_position_all.groupby(["condition_id", "market_label", "end_date_iso"], as_index=False, dropna=False)
    .agg(max_abs_usdc_position=("cum_usdc_position", lambda s: s.abs().max()))
)
top_markets_position = market_position_rank.sort_values("max_abs_usdc_position", ascending=False).head(TOP_MARKETS_POSITION).copy()
market_position_ts = market_position_all[market_position_all["condition_id"].isin(top_markets_position["condition_id"])].copy()

token_total_ts = _build_total_ts(granular_position_ts, "token_delta", "net_token_position", "cum_token_position")
usdc_total_ts = _build_total_ts(granular_position_ts, "usdc_delta", "net_usdc_position", "cum_usdc_position")

print(
    f"Prepared plot datasets: total pnl={len(pnl_total_ts)}, "
    f"wallet pnl series={wallet_pnl_ts['wallet'].nunique()}, "
    f"market pnl series={market_pnl_ts['condition_id'].nunique()}, "
    f"wallet position series={wallet_position_ts['wallet'].nunique()}, "
    f"market position series={market_position_ts['condition_id'].nunique()}"
)


Prepared plot datasets: total pnl=18063, wallet pnl series=10, market pnl series=10, wallet position series=10, market position series=10


In [313]:
fig_pnl = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total cumulative PnL",
        f"Per-wallet cumulative PnL (top {TOP_WALLETS_PNL} by abs total PnL)",
        f"Per-market cumulative PnL (top {TOP_MARKETS_PNL} by abs total PnL)",
    ),
)

fig_pnl.add_trace(
    go.Scatter(
        x=pnl_total_ts["dt"],
        y=pnl_total_ts["cum_pnl_usdc"],
        mode="lines",
        name="Total",
        line=dict(width=3, color="#1f77b4"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Total cum PnL: %{y:,.2f} USDC<extra></extra>",
    ),
    row=1,
    col=1,
)

wallet_totals_map = top_wallets_pnl.set_index("wallet")["total_pnl_usdc"].to_dict()
for wallet in top_wallets_pnl["wallet"]:
    sub = wallet_pnl_ts[wallet_pnl_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    total = wallet_totals_map[wallet]
    fig_pnl.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_pnl_usdc"],
            mode="lines",
            name=f"{label} ({total:,.1f})",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Cum PnL: %{y:,.2f} USDC<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

market_totals_map = top_markets_pnl.set_index("condition_id")["total_pnl_usdc"].to_dict()
for condition_id in top_markets_pnl["condition_id"]:
    sub = market_pnl_ts[market_pnl_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    total = market_totals_map[condition_id]
    fig_pnl.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_pnl_usdc"],
            mode="lines",
            name=f"{label} ({total:,.1f})",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Cum PnL: %{y:,.2f} USDC<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_pnl.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_pnl.update_yaxes(title_text="USDC", row=1, col=1)
fig_pnl.update_yaxes(title_text="USDC", row=2, col=1)
fig_pnl.update_yaxes(title_text="USDC", row=3, col=1)
fig_pnl.update_xaxes(title_text="Time", row=3, col=1)
fig_pnl.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - cumulative PnL",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_pnl.show(renderer="browser")


In [314]:
fig_token_pos = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total signed token position",
        f"Per-wallet signed token position (top {TOP_WALLETS_POSITION} by max abs USDC position)",
        f"Per-market signed token position (top {TOP_MARKETS_POSITION} by max abs USDC position)",
    ),
)

fig_token_pos.add_trace(
    go.Scatter(
        x=token_total_ts["dt"],
        y=token_total_ts["cum_token_position"],
        mode="lines",
        line_shape="hv",
        name="Total token position",
        line=dict(width=3, color="#2ca02c"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Signed token position: %{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

for wallet in top_wallets_position["wallet"]:
    sub = wallet_position_ts[wallet_position_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_token_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_token_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f} USDC)",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed token position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

for condition_id in top_markets_position["condition_id"]:
    sub = market_position_ts[market_position_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_token_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_token_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f} USDC)",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed token position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_token_pos.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_token_pos.update_yaxes(title_text="Tokens", row=1, col=1)
fig_token_pos.update_yaxes(title_text="Tokens", row=2, col=1)
fig_token_pos.update_yaxes(title_text="Tokens", row=3, col=1)
fig_token_pos.update_xaxes(title_text="Time", row=3, col=1)
fig_token_pos.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - signed token positions",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_token_pos.show(renderer="browser")


In [315]:
fig_usdc_pos = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total signed USDC position (using traded price)",
        f"Per-wallet signed USDC position (top {TOP_WALLETS_POSITION} by max abs USDC position)",
        f"Per-market signed USDC position (top {TOP_MARKETS_POSITION} by max abs USDC position)",
    ),
)

fig_usdc_pos.add_trace(
    go.Scatter(
        x=usdc_total_ts["dt"],
        y=usdc_total_ts["cum_usdc_position"],
        mode="lines",
        line_shape="hv",
        name="Total USDC position",
        line=dict(width=3, color="#d62728"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Signed USDC position: %{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

for wallet in top_wallets_position["wallet"]:
    sub = wallet_position_ts[wallet_position_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_usdc_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_usdc_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f})",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed USDC position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

for condition_id in top_markets_position["condition_id"]:
    sub = market_position_ts[market_position_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_usdc_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_usdc_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f})",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed USDC position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_usdc_pos.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_usdc_pos.update_yaxes(title_text="USDC", row=1, col=1)
fig_usdc_pos.update_yaxes(title_text="USDC", row=2, col=1)
fig_usdc_pos.update_yaxes(title_text="USDC", row=3, col=1)
fig_usdc_pos.update_xaxes(title_text="Time", row=3, col=1)
fig_usdc_pos.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - signed USDC positions",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_usdc_pos.show(renderer="browser")
